## Problem: Add Grad-CAM to Explain Model Predictions

### Problem Statement
Use Grad-CAM (Gradient-weighted Class Activation Mapping, Selvaraju et al., 2017) to explain the predictions of a convolutional neural network (here a pretrained ResNet-18). Grad-CAM highlights the regions of the input image that contribute most to the score of a chosen class.

For a target class $c$ with pre-softmax score (logit) $y^c$, and the $K$ feature maps $A^k$ of a late convolutional layer:
$$
\alpha_k^c = \frac{1}{Z} \sum_{i,j} \frac{\partial y^c}{\partial A^k_{ij}}, \qquad
L^c_\text{Grad-CAM} = \operatorname{ReLU}\Big(\sum_k \alpha_k^c A^k\Big)
$$
The channel weights $\alpha_k^c$ are the global-average-pooled gradients; the ReLU keeps only regions with a *positive* influence on class $c$. The map has the layer's spatial size (7×7 for ResNet-18's last block), so it is upsampled to the image size for display.

### Requirements

1. **Hooks** (provided): a forward hook on the target layer — the last residual block, `model.layer4[-1]` — saves its output activations $A$ and registers a hook on that output tensor to save $\partial y^c / \partial A$ during the backward pass.

2. **Forward pass**: run the preprocessed image through the model to get the logits.
    ```
    output = ...
    ```

3. **Backward pass**: pick the predicted class (the highest logit) and backpropagate **its logit** — not the softmax probability, which mixes in every other class.
    ```
    predicted_class = ...
    model.zero_grad()
    ...
    ```

4. **Grad-CAM heatmap**: average the gradients over the spatial dimensions to get one weight per channel, take the weighted sum of the activation maps over channels, and apply a ReLU.
    ```
    weights = ...
    heatmap = ...
    ```

In [ ]:
# Explain a CNN model's predictions using Grad-CAM in PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
# Load a pre-trained ResNet model
weights_enum = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights_enum)
model.eval()

# Variables filled in by the hooks
gradients = None
activations = None

def save_gradients(grad):
    global gradients
    gradients = grad  # dy^c / dA, same shape as A

def save_activations(module, input, output):
    global activations
    activations = output.detach()          # A: (1, K, h, w)
    output.register_hook(save_gradients)   # called with dy^c/dA during backward

# Hook the last residual block: the last spatial feature maps before global pooling.
# (A tensor hook on the output is the robust way to get dy/dA; the old
# module.register_backward_hook is deprecated and can report the wrong tensors.)
target_layer = model.layer4[-1]
hook_handle = target_layer.register_forward_hook(save_activations)

In [ ]:
# Fetch a sample image: a real photo makes the heatmap meaningful.
# Falls back to random noise if there is no internet connection.
import io
import urllib.request

url = "https://github.com/pytorch/hub/raw/master/images/dog.jpg"
try:
    with urllib.request.urlopen(url, timeout=10) as f:
        image = Image.open(io.BytesIO(f.read())).convert("RGB")
except Exception as e:
    print(f"Could not download the sample image ({e}); using random noise instead")
    image, _ = datasets.FakeData(size=1, image_size=(3, 224, 224))[0]  # a PIL image

# Preprocess the image for the model (ImageNet statistics)
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
input_tensor = preprocess(image).unsqueeze(0)

# Perform a forward pass
output = model(input_tensor)                     # (1, 1000) logits
predicted_class = output.argmax(dim=1).item()

# Perform a backward pass for the predicted class: the gradient of its logit
model.zero_grad()
output[0, predicted_class].backward()

In [ ]:
# Generate Grad-CAM heatmap
weights = gradients.mean(dim=(2, 3), keepdim=True)                       # alpha_k: (1, K, 1, 1)
heatmap = torch.relu((weights * activations).sum(dim=1, keepdim=True))   # (1, 1, h, w)

# Upsample to the image size (PIL size is (width, height)) and normalize to [0, 1]
heatmap = F.interpolate(heatmap, size=(image.size[1], image.size[0]), mode="bilinear", align_corners=False)
heatmap = heatmap[0, 0]
heatmap = heatmap / (heatmap.max() + 1e-8)  # eps: the map can be all zero

In [ ]:
# Display the image with the Grad-CAM heatmap
class_name = weights_enum.meta["categories"][predicted_class]
plt.imshow(image)
plt.imshow(heatmap.numpy(), alpha=0.5, cmap='jet')
plt.title(f"Predicted class: {class_name} ({predicted_class})")
plt.axis('off')
plt.show()

hook_handle.remove()  # detach the hook once done